# 03 - Heston Pricing and Calibration

The Heston stochastic-volatility model priced two ways, Monte Carlo and a characteristic-
function integral, validated against the MScFE 620 group-project numbers. The model
produces a volatility smile, and its five parameters $\{v_0, \kappa, \theta, \xi, \rho\}$
are recovered by calibration to an implied-volatility surface.


## Setup

In [ ]:
!pip install -q numpy scipy matplotlib pandas

import subprocess, os, sys
from google.colab import userdata

GITHUB_USER = "your-username"   # <-- edit this
REPO = "derivatives-pricing"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)

!pip install -q -e .
src = os.path.abspath("src")
if src not in sys.path:
    sys.path.insert(0, src)
import derivpricing
print("derivpricing", derivpricing.__version__)

## Two pricers, one model

The Euler Monte Carlo scheme and the characteristic-function pricer value the same Heston
dynamics. At the group-project parameters ($S_0=80$, $v_0=0.032$, $\kappa=1.85$,
$\theta=0.045$, $\xi=0.35$, $T=0.25$), they agree to Monte Carlo error and reproduce the
project's reported call prices of $3.48$ at $\rho=-0.30$ and $3.50$ at $\rho=-0.70$.


In [ ]:
from derivpricing.heston import heston_cf_price, heston_mc
p = dict(S0=80, K=80, T=0.25, r=0.055, v0=0.032, kappa=1.85, theta=0.045, xi=0.35)
for rho, project in [(-0.30, 3.48), (-0.70, 3.50)]:
    cf = heston_cf_price(rho=rho, option_type="call", **p)
    mc, se = heston_mc(rho=rho, option_type="call", **p)
    print(f"rho={rho}:  char-function {cf:.4f}   Monte Carlo {mc:.4f} (SE {se:.3f})   project {project}")

## The volatility smile

Pricing across strikes with the fast characteristic-function pricer and inverting to implied
volatility traces the smile. A more negative $\rho$ steepens the skew, since down moves in
the spot pair with up moves in variance.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from derivpricing.analytic import implied_vol

S0, r, T = 80.0, 0.055, 0.25
strikes = np.linspace(64, 100, 25)
base = dict(v0=0.032, kappa=1.85, theta=0.045, xi=0.35)

plt.figure(figsize=(8, 4.5))
for rho in (-0.30, -0.70):
    ivs = []
    for Kx in strikes:
        price = heston_cf_price(S0, Kx, T, r, rho=rho, option_type="call", **base)
        iv, _ = implied_vol(price, S0, Kx, T, r, "call")
        ivs.append(iv)
    plt.plot(strikes / S0, ivs, "o-", label=f"rho = {rho}", alpha=0.8)
plt.xlabel("moneyness K / S0"); plt.ylabel("implied volatility")
plt.title("Heston implied-volatility smile"); plt.legend(); plt.tight_layout(); plt.show()

## Calibrating to a market surface

Here a surface is generated from known Heston parameters, then recovered by least squares.
Fitting synthetic data confirms the machinery; on real quotes the same routine returns the
parameters plus the residuals that show how well the model fits.


In [ ]:
from derivpricing.calibration import implied_vol_surface, calibrate_heston

true = dict(v0=0.032, kappa=1.85, theta=0.045, xi=0.35, rho=-0.50)
strikes = [S0/m for m in (0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15)]
mats = [0.25, 0.5, 1.0]
price_fn = lambda Kx, Tx: heston_cf_price(S0, Kx, Tx, r, option_type="call", **true)
market = implied_vol_surface(S0, r, strikes, mats, price_fn)

res = calibrate_heston(S0, r, market, x0=[0.05, 3.0, 0.05, 0.5, -0.2])
print(f"success {res['success']}   RMSE {res['rmse']:.2e} vol points   max|resid| {res['max_abs_residual']:.2e}")
import pandas as pd
pd.DataFrame({"fitted": res["params"], "true": true}).round(4)

## Fitted smile over the market smile

Overlaying the calibrated model's implied vols on the market smile at the shortest maturity
shows the fit visually, with the residuals kept small across strikes.


In [ ]:
T0 = 0.25
ks = sorted(k for (k, t) in market if abs(t - T0) < 1e-9)
mkt_iv = [market[(k, T0)] for k in ks]
fit = res["params"]
mdl_iv = []
for k in ks:
    price = heston_cf_price(S0, k, T0, r, fit["v0"], fit["kappa"], fit["theta"], fit["xi"], fit["rho"], "call")
    iv, _ = implied_vol(price, S0, k, T0, r, "call")
    mdl_iv.append(iv)

plt.figure(figsize=(8, 4.5))
plt.plot(np.array(ks)/S0, mkt_iv, "o", ms=9, label="market", alpha=0.7)
plt.plot(np.array(ks)/S0, mdl_iv, "x-", label="calibrated Heston")
plt.xlabel("moneyness K / S0"); plt.ylabel("implied volatility")
plt.title(f"Calibrated fit at T = {T0}"); plt.legend(); plt.tight_layout(); plt.show()

## Recap

The two pricers agree and match the group-project prices, the model reproduces the smile,
and calibration recovers the parameters with small residuals. Together with `01` and `02`,
the library now covers analytic, lattice, Monte Carlo, and stochastic-volatility pricing
with calibration, each demonstrated with its own check.
